In [ ]:
import pandas as pd

df = pd.read_parquet("data/processed/moves.parquet")
print(len(df), df.game_id.nunique())
print(df.split.value_counts(normalize=True))
print("blunder rate:", (df.wp_drop >= 0.15).mean())
print(df.wp_drop.describe())

# Does blunder rate rise as the clock falls?
bins = pd.cut(df.clock_before, [0, 30, 60, 120, 300, 601])
print(df.groupby(bins, observed=True).wp_drop.apply(lambda s: (s >= 0.15).mean()))
print(df.mover_elo.describe())

count    6.587942e+06
mean     1.543167e+03
std      3.900451e+02
min      4.000000e+02
25%      1.260000e+03
50%      1.549000e+03
75%      1.831000e+03
max      3.046000e+03
Name: mover_elo, dtype: float64


In [4]:
import pandas as pd

df = pd.read_parquet("data/processed/moves.parquet")
bins = [0, 30, 60, 120, 300, 601]

# Check 1: do the clocks behave like clocks?
print((df.time_spent < 0).mean())  # should be ~0 at 10+0 (no increment)
mono = df.groupby(["game_id", "mover_is_white"]).clock_before.apply(lambda s: s.is_monotonic_decreasing)
print(mono.mean())  # should be ~1.0
print(df.groupby(pd.cut(df.clock_before, bins), observed=True).agg(
    rows=("ply", "size"),
    mean_ply=("ply", "mean"),
    mean_time_spent=("time_spent", "mean"),
    mean_wp_before=("wp_before", "mean"),
))

# Check 2: blunder rate where a blunder is actually possible
elig = df[(df.ply > 10) & df.wp_before.between(0.25, 0.75)]
print(elig.groupby(pd.cut(elig.clock_before, bins), observed=True).wp_drop.agg(
    rows="size", blunder_rate=lambda s: (s >= 0.15).mean()
))

0.00010230812596710778
0.99826
                 rows    mean_ply  mean_time_spent  mean_wp_before
clock_before                                                      
(0, 30]         87376  106.040526         2.145349        0.529808
(30, 60]        93294   94.169143         4.099781        0.535033
(60, 120]      159984   87.838671         7.051093        0.524414
(120, 300]     853824   72.718321         9.872489        0.516507
(300, 601]    5391924   31.390389         8.547273        0.518317
                 rows  blunder_rate
clock_before                       
(0, 30]         21397      0.140113
(30, 60]        25703      0.130957
(60, 120]       46342      0.124919
(120, 300]     275610      0.121465
(300, 601]    2598873      0.093797


In [28]:
first_move_df = df[df["ply"] <= 2]
print(first_move_df[first_move_df.time_spent < 301]["time_spent"].shape)

(200000,)


In [22]:
elig = df[(df.ply > 10) & df.wp_before.between(0.25, 0.75)].copy()
elig["clock_bin"] = pd.cut(elig.clock_before, [0, 60, 300, 601])
elig["ply_bin"] = pd.cut(elig.ply, [10, 30, 50, 80, 300])
print(elig.groupby(["ply_bin", "clock_bin"], observed=True).wp_drop.agg(
    rows="size", blunder_rate=lambda s: (s >= 0.15).mean()
).unstack())

             rows                      blunder_rate                     
clock_bin (0, 60] (60, 300] (300, 601]      (0, 60] (60, 300] (300, 601]
ply_bin                                                                 
(10, 30]       85     16401    1553374     0.129412  0.059509   0.075323
(30, 50]     1143     57895     749130     0.193351  0.138561   0.119700
(50, 80]    13695    152956     258509     0.168310  0.135000   0.127125
(80, 300]   32177     94700      37860     0.118936  0.101573   0.111675
